In [ ]:
import json

from fastapi.encoders import jsonable_encoder


from app.infra.database import AsyncSessionFactory
from app.modules.product.repository import ProductRepository

"""
    创建产品数据层对象
    拿到每个渲染的产品数据库对象
"""
# 创建session,拿到产品数据层对象
async with AsyncSessionFactory() as session:
    product_repository = ProductRepository(session)

    product_result_list = await product_repository.get_product_list_repository(category=None)

    product = product_result_list[0]
    # 打印返回的orm对象结果
    print(json.dumps(jsonable_encoder(product),indent=4,ensure_ascii=False))



In [2]:
# 通过检索出的文件名,查找到文件,将文件喂给mineru
from mineru import MinerU
from app.core.config import settings
import os

"""
    拿着产品里面的文档名称,去检索对应的pdf文件名
"""
file_path = os.path.join('../../app/data/raw/kb', product.clause_name)

# 创建转换器客户端
mineru_client = MinerU(settings.rag.miner_u_token)

# 通过客户端检索文件
result = mineru_client.extract(file_path)

# 将转换后的文档存入文件中
if result.markdown is not None:
    # 文件写入为md文档
    with open('md/md_result1.md','w',encoding='utf-8') as file:
        file.write(result.markdown)

In [7]:
"""
    通过大模型对markdown文档做归类
"""
from langchain.chat_models import init_chat_model

MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()

llm = init_chat_model(
    "deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}},
    max_tokens=200000
)

response = llm.invoke([
    {"role": "system", "content": MARKDOWN_OPTIMIZATION_PROMPT},
    {"role": "user", "content": result.markdown},
])

# 拿到转换后的markdown文档
markdown = response.text

In [10]:
#  二次写入
if markdown is not None:
    # 文件写入为md文档
    with open('md/md_result2.md','w',encoding='utf-8') as file:
        file.write(markdown)

In [13]:
# 创建markdown切分器
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

markdown_splitter = MarkdownHeaderTextSplitter(
    # 按照标题拆分四级
     headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
        ("####", "h4"),
    ],
    strip_headers=True,  # 去除标题头部(后期需要手动将多级标题拼接到正文内容中)
)

# 对markdown文本做切分
sections = markdown_splitter.split_text(markdown)


In [14]:
# 创建子块切分器,子块使用递归切分器
child_splitter = RecursiveCharacterTextSplitter(
    # 定义切分规则
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "；", ";", "，", ","],
    keep_separator="end",
)

In [15]:
from app.rag.models import ParentChunk
from uuid import uuid4
from langchain_core.documents import Document

# 定义转换后的结果集列表,后期将父块批量添加入postgresql
parent_chunks: list[ParentChunk] = []
# 子块结果列表,创建document对象,存入向量数据库
child_chunks: list[Document] =  []

# 循环拆分出来的对象,做数据转换
for section in sections:
    # 每一个section是document对象,也就是父块的原数据来源

    # 创建表对象
    # section_path指的就是从根标题找到文章正文的路径
    section_path = section.metadata.values()
    parent_chunk = ParentChunk(
        id=uuid4(),
        product_id=product.id,
        clause_name=product.clause_name,
        section_path=section_path,
        content=section.page_content
    )
    # 将单个表对象存入列表,后续做批量新增
    parent_chunks.append(parent_chunk)

    # 接下来对子块做切分
    # 传入切分好的父块doc对象
    section_child_chunks = child_splitter.split_documents([section])
    for child_chunk in section_child_chunks:
        # 修改子chunk的对象属性
        child_chunk.metadata = {
              # 关联父块id
             'parent_id': str(parent_chunk.id),
             'product_id': product.id    # 关联产品id
        }
        # 修复每一个子块的所属,也就是它的标题
        child_chunk.page_content = '\n'.join(section_path)+ '\n' + child_chunk.page_content
    # 循环结束后,将每一个子chunk添加进列表中
    child_chunks.extend(section_child_chunks)



In [16]:
"""
    将父块写入postgresql数据库
"""
from app.rag.repository import ParentChunkRepository

# 创建对象
async with AsyncSessionFactory() as session:
    async with session.begin():
        # 开启事务,创建对象
        parent_chunk_repository = ParentChunkRepository(session)
        # 先删除数据库关于此产品的全部父chunk
        await parent_chunk_repository.delete_parent_chunks(product.id)
        # 新增父chunk
        await parent_chunk_repository.add_parent_chunks(parent_chunks)

2026-09-06 16:02:39,155 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-06 16:02:39,158 INFO sqlalchemy.engine.Engine DELETE FROM parent_chunks WHERE parent_chunks.product_id = $1::BIGINT
2026-09-06 16:02:39,158 INFO sqlalchemy.engine.Engine [generated in 0.00059s] (1,)
2026-09-06 16:02:39,172 INFO sqlalchemy.engine.Engine INSERT INTO parent_chunks (id, product_id, clause_name, section_path, content) VALUES ($1::UUID, $2::BIGINT, $3::VARCHAR, $4::TEXT[], $5::VARCHAR), ($6::UUID, $7::BIGINT, $8::VARCHAR, $9::TEXT[], $10::VARCHAR), ($11::UUID, $12::BIGINT, $13::VARCHAR,  ... 24937 characters truncated ...  $1703::VARCHAR, $1704::TEXT[], $1705::VARCHAR) RETURNING parent_chunks.created_at, parent_chunks.id
2026-09-06 16:02:39,173 INFO sqlalchemy.engine.Engine [generated in 0.00065s (insertmanyvalues) 1/1 (ordered)] (UUID('fd5da5cd-e3d4-4601-b8e2-3901d9fc811d'), 1, '信美相互星辰守护B款少儿重疾险条款.pdf', ['信美相互互联网星辰守护B款少儿重大疾病保险条款'], '在本条款中，“您”指投保人，“我们”指信美人寿相互保险社，“本合同”指您与我们之间订立的“信美相互互联网星辰守护B款少儿重大疾病保

In [17]:
from langchain_ollama import OllamaEmbeddings
# 初始化向量数据库客户端
from langchain_milvus import Milvus, BM25BuiltInFunction

"""
    将子块写入向量数据库
"""
# 创建ollama向量模型对象

ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

# 创建向量数据库客户端
milvus_client = Milvus(
    embedding_function=ollama_embeddings,  # 稠密向量模型
    collection_name="insurance_collection",  # collection名称
    builtin_function=BM25BuiltInFunction(  # 生成稀疏向量的函数
        analyzer_params={"type": "chinese"}  # 指定中文分词
    ),
    vector_field=["dense", "sparse"],  # 向量字段，包括稠密和稀疏
    connection_args={
        "uri": settings.rag.milvus_url,  # milvus的uri路径
    },
    drop_old=False,  # 是否删除旧的collection，避免重复创建
    # 自动主键
    auto_id=False
)

# 做20个一批的写入操作
batch_documents = [
    child_chunks[i:i+20]
    for i in range(0,len(child_chunks),20)
]
# 写入操作
for batch in  batch_documents:
    milvus_client.add_documents(batch)

C:\Users\84370\Desktop\黑马-python阶段\python-project\wcy-insurance\agent-service\.venv\Lib\site-packages\langchain_milvus\vectorstores\milvus.py:1408: UserWarning: No ids provided and auto_id is False. Setting auto_id to True automatically.
  warnings.warn(
